# model

> Build a segmentation model (any `segmentation_models_pytorch` architecture + encoder) with configurable `in_channels` and `num_classes`, plus helpers for differential learning rates and encoder freezing.

We use `segmentation_models_pytorch`, which adapts the pretrained encoder's first convolution to any `in_channels` and lets us set `classes` to 1 (binary) or N (multi-class) directly.


In [ ]:
#| default_exp model

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import List, Optional

import torch
import torch.nn as nn



In [ ]:
#| export
import segmentation_models_pytorch as smp

## Build the model

`architecture` selects the smp segmentation head (`unet`, `fpn`, `deeplabv3plus`, …). `arch` / `encoder` selects any smp encoder backbone (`mobilenet_v2`, `resnet34`, `efficientnet-b3`, …).

`num_classes == 1` produces a single logit channel for sigmoid; `num_classes > 1` produces one channel per class for softmax. `activation=None` keeps the head returning raw logits, which is what our losses and metrics expect.


In [ ]:
#| eval: false
# scratch: only runs where a local weights cache is present
from pathlib import Path
weights_path = Path(r'/path/to/smp_models/mobilenet_v2.pth')
pretrained = True
use_imagenet  = pretrained and weights_path is None
print(f'pretrained={pretrained},\n use_imagenet={use_imagenet},\n weights_path={weights_path}\n')

In [ ]:
#| eval: false
in_channels = 3 
num_classes = 1
model = smp.Unet(
        encoder_name='mobilenet_v2',
        encoder_weights='imagenet' if use_imagenet else None,
        in_channels=in_channels,
        classes=num_classes,
        activation=None,
    )
state = torch.load(weights_path, map_location='cpu')

In [ ]:
#| eval: false
res = nn.Module.load_state_dict(
    model.encoder, 
    state, 
    strict=False)

In [ ]:
#| eval: false
missing, unexpected = res.missing_keys, res.unexpected_keys
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

In [ ]:
#| eval: false
missing, unexpected = model.encoder.load_state_dict(state, strict=False)

In [ ]:
import torch; print(torch.__version__)

In [ ]:
#| export
SMP_ARCHITECTURES = {
    'unet': smp.Unet,
    'unetplusplus': smp.UnetPlusPlus,
    'manet': smp.MAnet,
    'linknet': smp.Linknet,
    'fpn': smp.FPN,
    'pspnet': smp.PSPNet,
    'pan': smp.PAN,
    'deeplabv3': smp.DeepLabV3,
    'deeplabv3plus': smp.DeepLabV3Plus,
    'upernet': smp.UPerNet,
}


def list_smp_architectures() -> List[str]:
    """Segmentation head names accepted by `get_model(architecture=...)`."""
    return sorted(SMP_ARCHITECTURES.keys())


def list_smp_encoders() -> List[str]:
    """Encoder / backbone names from segmentation-models-pytorch (includes registered `dinov3_*`)."""
    return sorted(smp.encoders.get_encoder_names())


def _normalize_architecture(name: str) -> str:
    key = name.lower().replace('-', '').replace('_', '')
    aliases = {'unetplusplus': 'unetplusplus', 'deeplabv3plus': 'deeplabv3plus'}
    return aliases.get(key, key)


def _load_local_encoder_weights(model: nn.Module, weights_path: str, device: str) -> None:
    state = torch.load(weights_path, map_location=device, weights_only=False)
    model_state = model.encoder.state_dict()
    compatible = {
        k: v for k, v in state.items()
        if k in model_state and v.shape == model_state[k].shape
    }
    skipped = [
        k for k in state
        if k in model_state and state[k].shape != model_state[k].shape
    ]
    if skipped:
        print(
            f'WARNING: {len(skipped)} key(s) skipped from {weights_path} due to shape mismatch '
            f'(checkpoint in_channels ≠ model in_channels): {skipped}'
        )
    res = model.encoder.load_state_dict(compatible, strict=False)
    print(f'Loaded encoder from {weights_path} ({len(compatible)}/{len(model_state)} keys matched)')
    if res.missing_keys:
        print(f'  missing keys: {len(res.missing_keys)} (e.g. {res.missing_keys[0]})')
    if res.unexpected_keys:
        print(f'  unexpected keys: {len(res.unexpected_keys)} (e.g. {res.unexpected_keys[0]})')


def get_model(
    arch: str = 'mobilenet_v2',   # encoder name — alias kept for Hydra `model.arch`
    in_channels: int = 3,
    num_classes: int = 1,         # 1 = binary, >1 = multi-class
    pretrained: bool = True,      # ImageNet weights (ignored when weights_path is set)
    weights_path: Optional[str] = None,  # local encoder .pth from download_encoder_weights
    device: str = 'cpu',
    architecture: str = 'unet',   # smp head: unet | fpn | deeplabv3plus | ...
    encoder: Optional[str] = None,  # explicit encoder name; overrides `arch` when set
) -> nn.Module:
    """Build an smp model returning raw logits `(B, num_classes, H, W)`.

    Pass `weights_path` on air-gapped HPC instead of downloading ImageNet weights.
    Use `list_smp_architectures()` and `list_smp_encoders()` to discover valid names.
    `encoder='dinov3_vit_small'` / `'dinov3_vit_base'` build a DPT-reassembled DINOv3
    ViT encoder (see the DINOv3 section below) instead of a CNN backbone.

    Example:
        >>> m = get_model('resnet34', architecture='fpn', in_channels=3, num_classes=1, pretrained=False)
        >>> tuple(m(torch.randn(1, 3, 64, 64)).shape)
        (1, 1, 64, 64)
    """
    encoder_name = encoder if encoder is not None else arch
    arch_key = _normalize_architecture(architecture)
    if arch_key not in SMP_ARCHITECTURES:
        raise ValueError(
            f"Unknown architecture {architecture!r}. "
            f"Choose one of: {', '.join(list_smp_architectures())}"
        )
    valid_encoders = smp.encoders.get_encoder_names()
    if encoder_name not in valid_encoders:
        raise ValueError(
            f"Unknown encoder {encoder_name!r}. "
            f"Use list_smp_encoders() — {len(valid_encoders)} encoders available."
        )
    use_imagenet = pretrained and weights_path is None
    model_cls = SMP_ARCHITECTURES[arch_key]
    if encoder_name.startswith('dinov3_'):
        # DINOv3 weights are downloaded inside Dinov3ReassembleEncoder.__init__ via
        # timm's own hub, not smp's encoder_weights='imagenet' state-dict loader (which
        # would `load_state_dict` the *whole* encoder, including our reassemble layers,
        # against a checkpoint that only has the plain ViT backbone -> missing keys).
        # Thread `pretrained` through the registry's construction params instead.
        smp.encoders.encoders[encoder_name]['params']['pretrained'] = use_imagenet
        encoder_weights = None
    else:
        encoder_weights = 'imagenet' if use_imagenet else None
    model = model_cls(
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        in_channels=in_channels,
        classes=num_classes,
        activation=None,
    )
    if weights_path is not None:
        _load_local_encoder_weights(model, weights_path, device)
    return model


In [ ]:
# test: output channels follow num_classes for binary and multi-class
m1 = get_model('mobilenet_v2', in_channels=2, num_classes=1, pretrained=False)
assert tuple(m1(torch.randn(1, 2, 64, 64)).shape) == (1, 1, 64, 64)
m3 = get_model('mobilenet_v2', in_channels=4, num_classes=3, pretrained=False)
assert tuple(m3(torch.randn(1, 4, 64, 64)).shape) == (1, 3, 64, 64)
m_fpn = get_model('resnet18', architecture='fpn', in_channels=3, num_classes=1, pretrained=False)
assert tuple(m_fpn(torch.randn(1, 3, 64, 64)).shape) == (1, 1, 64, 64)
assert 'resnet34' in list_smp_encoders()
assert 'deeplabv3plus' in list_smp_architectures()


In [ ]:
#| hide
#| eval: false
from pathlib import Path
weights_path =Path(r'/path/to/smp_models/mobilenet_v2.pth')
print(f' Weights path exist: {weights_path.exists()}')
m1 = get_model(
    'mobilenet_v2', 
    in_channels=3, 
    num_classes=1, 
    pretrained=True, 
    weights_path=weights_path)



In [ ]:
#| eval: false
assert tuple(m1(torch.randn(1, 2, 64, 64)).shape) == (1, 1, 64, 64)

## Offline / air-gapped HPC usage

On an HPC without internet, `encoder_weights='imagenet'` will raise a download error.  
The workflow is:

| Step | Where | Command |
|------|-------|---------|
| 1 | Any machine **with** internet | `download_encoder_weights('resnet34', '/tmp/resnet34.pth')` |
| 2 | Copy the `.pth` file to HPC | `scp /tmp/resnet34.pth hpc:/data/weights/` |
| 3 | HPC (no internet) | `get_model('resnet34', weights_path='/data/weights/resnet34.pth')` |

`download_encoder_weights` lets smp download + cache the weights normally, then saves **just the encoder state-dict** to a portable file.  
`get_model(weights_path=...)` builds the model with `encoder_weights=None` and loads that file — no network call is made.

In [ ]:
#| export
def download_encoder_weights(
    encoder_name: str,   # smp encoder name  (e.g. 'resnet34', 'efficientnet-b0')
    save_path: str,      # local .pth path to write   (e.g. '/data/weights/resnet34.pth')
) -> None:
    """Download ImageNet encoder weights to a portable local file.

    Run **once** on any machine that has internet access, copy the file to your
    air-gapped HPC, then use `get_model(weights_path=...)`.

    Internally builds a full smp.Unet to trigger the normal smp/timm download,
    then extracts and saves only the encoder state-dict — so the file is small
    (~20-100 MB) and independent of your `in_channels` / `num_classes` settings.

    Example:
        >>> import tempfile, os
        >>> p = os.path.join(tempfile.mkdtemp(), 'enc.pth')
        >>> download_encoder_weights('mobilenet_v2', p)   # doctest: +SKIP
        >>> os.path.exists(p)                              # doctest: +SKIP
        True
    """
    from pathlib import Path
    tmp = smp.Unet(encoder_name=encoder_name, encoder_weights='imagenet',
                   in_channels=3, classes=1, activation=None)
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(tmp.encoder.state_dict(), save_path)
    n = len(tmp.encoder.state_dict())
    size_mb = Path(save_path).stat().st_size / 1e6
    print(f'Saved {encoder_name} encoder  ({n} tensors, {size_mb:.1f} MB)  →  {save_path}')
    del tmp

In [ ]:
#| eval: false
# What encoders are available in segmentation_models_pytorch (smp)?
import segmentation_models_pytorch as smp

# List all available encoders (common in stable SMP releases)
print("Available SMP encoders:")
for name in sorted(smp.encoders.get_encoder_names()):
    print("-", name)
    encoder_name = name
    try:
        download_encoder_weights(encoder_name, f'x:/Models/smp_models/{encoder_name}.pth')
    except Exception as e:
        print(f"Error downloading {encoder_name}: {e}")
    


In [ ]:
#| eval: false

# Example usage: pick one and download its weights
encoder_name = 'resnet18'  # or choose any from the printed list
download_encoder_weights(encoder_name, f'x:/Models/smp_models/{encoder_name}.pth')

## Differential learning rates and freezing

The pretrained encoder needs a gentler learning rate than the randomly-initialised decoder. Freezing the encoder for a few warm-up epochs lets the decoder catch up before fine-tuning the whole network.

In [ ]:
#| export
def get_param_groups(
    model: nn.Module,        # an smp model (or any model; falls back to one group if no .encoder)
    encoder_lr: float = 1e-4,  # LR for the pretrained encoder
    decoder_lr: float = 1e-3,  # LR for the decoder + head
) -> List[dict]:
    "Two parameter groups so encoder and decoder train at different rates."
    if not hasattr(model, 'encoder'):
        return [{'params': list(model.parameters()), 'lr': decoder_lr}]
    enc = list(model.encoder.parameters())
    dec = list(model.decoder.parameters()) + list(model.segmentation_head.parameters())
    return [{'params': enc, 'lr': encoder_lr}, {'params': dec, 'lr': decoder_lr}]

def freeze_encoder(model: nn.Module) -> None:
    "Freeze encoder weights for a decoder warm-up phase. No-op if model has no .encoder."
    if not hasattr(model, 'encoder'):
        return
    for p in model.encoder.parameters():
        p.requires_grad = False

def unfreeze_encoder(model: nn.Module) -> None:
    "Unfreeze the encoder after warm-up. No-op if model has no .encoder."
    if not hasattr(model, 'encoder'):
        return
    for p in model.encoder.parameters():
        p.requires_grad = True

## DINOv3 encoder-decoder

`list_smp_encoders()` only covers CNN-style backbones — there is no DINOv3 entry, because
DINOv3 is a plain Vision Transformer: `timm`'s `vit_*_patch16_dinov3` models emit a single
token grid at stride 16, not the stride `[2,4,8,16,32]` feature pyramid smp's `Unet`/`FPN`
decoders expect for their skip connections.

To use a DINOv3 backbone with those decoders anyway, we tap 4 evenly-spaced transformer
blocks (`timm`'s `forward_intermediates`) and reassemble each one to a different stride via
up/down-sampling convs — the same trick the DPT paper (Ranftl et al., 2021) uses to turn a
ViT into a dense-prediction backbone. ViT has no natural high-resolution feature, so the
missing stride-2 skip is synthesized by further upsampling the stride-4 map; this gives the
decoder a shallow skip for sharp boundaries instead of starting at stride 4.

The result is registered into `smp.encoders.encoders` under `dinov3_vit_small` /
`dinov3_vit_base`, so it plugs into the existing `get_model(arch=..., architecture='unet')`
call — no new API, no changes to `get_param_groups` / `freeze_encoder` / Hydra configs.

Pretrained DINOv3 weights come from `timm`'s own hub download (`pretrained=True` at
construction), not smp's `encoder_weights='imagenet'` path — that path expects to
`load_state_dict` a full encoder checkpoint with no extra reassemble layers, which would
raise missing-key errors here. `get_model()` accounts for this below.


In [ ]:
#| export
import timm
from segmentation_models_pytorch.encoders._base import EncoderMixin


class _Reassemble(nn.Module):
    "Project a ViT token feature map to `out_ch` channels, then resize to the target stride."
    def __init__(self, in_ch: int, out_ch: int, scale: float):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, out_ch, kernel_size=1)
        if scale > 1:
            self.resize = nn.Upsample(scale_factor=scale, mode='bilinear', align_corners=False)
        elif scale < 1:
            self.resize = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=round(1 / scale), padding=1)
        else:
            self.resize = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.resize(self.proj(x))


class Dinov3ReassembleEncoder(nn.Module, EncoderMixin):
    """DINOv3 ViT backbone + DPT-style reassemble, exposed as an smp `EncoderMixin`
    so it can be used as `get_model(arch='dinov3_vit_small', architecture='unet', ...)`.

    Forward returns `[x, stride2, stride4, stride8, stride16, stride32]` — 6 tensors for
    `depth=5`, matching what a CNN encoder (e.g. resnet34) would return.
    """

    def __init__(
        self,
        model_name: str = 'vit_small_patch16_dinov3',
        depth: int = 5,
        output_stride: int = 32,
        pretrained: bool = True,
        widths=(64, 96, 192, 384, 384),  # channel widths for stride 2,4,8,16,32
    ):
        super().__init__()
        self._depth = depth
        self._in_channels = 3
        self._output_stride = output_stride

        # Always built for 3-channel input; `set_in_channels()` (called by smp's
        # `get_encoder()` right after construction) adapts the patch-embed conv for
        # any other `in_channels`, reusing the same weight-averaging trick smp uses
        # for every other encoder.
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained, num_classes=0, dynamic_img_size=True,
        )
        embed_dim = self.backbone.embed_dim

        w2, w4, w8, w16, w32 = widths
        self.reassemble = nn.ModuleList([
            _Reassemble(embed_dim, w4, scale=4),    # earliest tap  -> stride 4
            _Reassemble(embed_dim, w8, scale=2),    # ..            -> stride 8
            _Reassemble(embed_dim, w16, scale=1),   # ..            -> stride 16
            _Reassemble(embed_dim, w32, scale=0.5), # last tap      -> stride 32
        ])
        # ViT has no native high-res feature -- synthesize stride 2 by upsampling
        # the stride-4 map instead of leaving the decoder's shallowest skip empty.
        self.stem_up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(w4, w2, kernel_size=3, padding=1),
        )
        self._out_channels = (3, w2, w4, w8, w16, w32)

    def set_in_channels(self, in_channels: int, pretrained: bool = True) -> None:
        "Adapt the patch-embed conv in place; `in_chans` can't be set after `timm.create_model`."
        if in_channels == 3:
            return
        from segmentation_models_pytorch.encoders._utils import patch_first_conv
        self._in_channels = in_channels
        self._out_channels = (in_channels,) + self._out_channels[1:]
        patch_first_conv(self.backbone, new_in_channels=in_channels, pretrained=pretrained)

    def forward(self, x: torch.Tensor) -> List[torch.Tensor]:
        feats = self.backbone.forward_intermediates(
            x, indices=4, output_fmt='NCHW', intermediates_only=True,
        )
        stride4, stride8, stride16, stride32 = [r(f) for r, f in zip(self.reassemble, feats)]
        stride2 = self.stem_up(stride4)
        return [x, stride2, stride4, stride8, stride16, stride32]


def _register_dinov3_encoders() -> None:
    "Register the DINOv3 encoders into smp so `list_smp_encoders()` and `get_model()` see them."
    for name, model_name in [
        ('dinov3_vit_small', 'vit_small_patch16_dinov3'),
        ('dinov3_vit_base', 'vit_base_patch16_dinov3'),
    ]:
        smp.encoders.encoders[name] = {
            'encoder': Dinov3ReassembleEncoder,
            'pretrained_settings': {},  # unused -- pretrained loading happens in __init__ via timm's hub
            'params': {'model_name': model_name, 'pretrained': True},
        }

_register_dinov3_encoders()


In [ ]:
# test: DINOv3 encoder is registered and pluggable through the normal get_model() call
assert {'dinov3_vit_small', 'dinov3_vit_base'} <= set(list_smp_encoders())

# small crop, no download -- pretrained=False just randomly inits the timm backbone
m_dino = get_model('dinov3_vit_small', architecture='unet', in_channels=3, num_classes=1, pretrained=False)
assert tuple(m_dino(torch.randn(1, 3, 64, 64)).shape) == (1, 1, 64, 64)

# single-channel input (grayscale X-ray, this project's common case) via set_in_channels
m_dino_gray = get_model('dinov3_vit_small', architecture='unet', in_channels=1, num_classes=1, pretrained=False)
assert tuple(m_dino_gray(torch.randn(1, 1, 64, 64)).shape) == (1, 1, 64, 64)
assert m_dino_gray.encoder.backbone.patch_embed.proj.in_channels == 1

# differential LR / freeze-unfreeze already work unchanged (encoder/decoder/segmentation_head present)
groups = get_param_groups(m_dino, encoder_lr=1e-5, decoder_lr=1e-3)
assert len(groups) == 2 and groups[0]['lr'] < groups[1]['lr']
freeze_encoder(m_dino)
assert all(not p.requires_grad for p in m_dino.encoder.parameters())
unfreeze_encoder(m_dino)
assert all(p.requires_grad for p in m_dino.encoder.parameters())


In [ ]:
# test
groups = get_param_groups(m1)
assert len(groups) == 2 and groups[0]['lr'] < groups[1]['lr']
freeze_encoder(m1)
assert all(not p.requires_grad for p in m1.encoder.parameters())
unfreeze_encoder(m1)
assert all(p.requires_grad for p in m1.encoder.parameters())

# fallback: a plain (non-SMP) model has no .encoder -- one param group, freeze/unfreeze are no-ops
toy = nn.Conv2d(3, 1, 3, padding=1)
toy_groups = get_param_groups(toy)
assert len(toy_groups) == 1 and toy_groups[0]['lr'] == 1e-3
assert set(toy_groups[0]['params']) == set(toy.parameters())
freeze_encoder(toy)   # no .encoder -> no-op, must not raise
unfreeze_encoder(toy)
assert all(p.requires_grad for p in toy.parameters())

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()